# Part 9

In [7]:
import re
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tokenizers import Tokenizer

In [1]:
import pandas as pd 
import time

PATH = '/media/trong-thanh/Data/University/Fourth Year/NLP/experiments/ex1/data.json'
df = pd.read_json(PATH, lines = True)
texts = df['text'].fillna('').tolist()

In [6]:
for index, i in enumerate(texts[:10]):
    print(f'Text {index}')
    print(i)

Text 0
Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.
The cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.
Text 1
Discussion in 'Mac OS X Lion (10.7)' started by axboi87, Jan 20, 2012.
I've got a 500gb internal drive and a 240gb SSD.
When trying to restore using disk utility i'm given the error "Not enough space on disk ____ to restore"
But I shouldn

## 9.1 Pipeline A Minimal

In [8]:
vector_1 = CountVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')
X_1 = vector_1.fit_transform(texts)

vocab_1 = len(vector_1.vocabulary_)
avg_tok_a = float(X_1.sum(axis=1).mean())
sparsity_a = 1.0 - (X_1.nnz / (X_1.shape[0] * X_1.shape[1]))


In [10]:
print(f'Vocabulary size: {vocab_1}')
print(f'Average token/docs: {avg_tok_a}')
print(f'Matrix shape: {X_1.shape}')
print(f'Matrix sparsity (S): {sparsity_a:.6f} ({sparsity_a * 100:.4f}%)')

Vocabulary size: 193837
Average token/docs: 369.6963666666667
Matrix shape: (30000, 193837)
Matrix sparsity (S): 0.999122 (99.9122%)


Pipeline A khá đơn giản Raw -> Lowercase -> Tokenization

Số từ vựng rất lớn so với 30000 văn bản, khoảng ~6 từ vựng / văn bản tức mỗi văn bản đóng góp trung bình 6 từ vựng.

Mức độ thưa rất lớn, spa 99.9122% tức tỉ lệ % các ô có giá trị bằng 0 trong các văn bản rất lớn. Tỷ lệ ô khác 0 = 100% − Sparsity = 100% − 99.9122% = 0.0878% nên mỗi văn bản trung bình chỉ chứa khoảng 170 từ vựng riêng biệt (không tính trùng lặp) trên tổng số 193.837 từ trong toàn bộ từ điển.

## 9.2 Pipeline B Normalized

In [11]:
import string 

regex = re.compile(f'[{re.escape(string.punctuation)}]')

def normalize(text):
    return regex.sub(' ', text.lower())

cleaned_texts_2 = [normalize(t) for t in texts]    

In [12]:
vector_2 = CountVectorizer(
    stop_words='english',         
    token_pattern=r'(?u)\b\w+\b'  
)
X_2 = vector_2.fit_transform(cleaned_texts_2)

In [13]:
vocab_b = len(vector_2.vocabulary_)
avg_tok_b = float(X_2.sum(axis=1).mean())
sparsity_b = 1.0 - (X_2.nnz / (X_2.shape[0] * X_2.shape[1]))

In [14]:
print(f'Vocabulary size: {vocab_b}')
print(f'Average token/docs: {avg_tok_b}')
print(f'Matrix shape: {X_2.shape}')
print(f'Matrix sparsity (S): {sparsity_b:.6f} ({sparsity_b * 100:.4f}%)')

Vocabulary size: 192764
Average token/docs: 197.4658
Matrix shape: (30000, 192764)
Matrix sparsity (S): 0.999351 (99.9351%)


Pipeline B thêm 2 bước so với A chuẩn hóa dấu câu và loại bỏ stop words.

Số từ vựng giảm từ 193.837 xuống 192.764 do loại bỏ stop words và làm sạch các từ dính dấu câu.

Số lượng token trung bình mỗi văn bản giảm mạnh từ ~370 xuống ~197 token (giảm gần 50%), cho thấy stop words chiếm phần lớn số lượng từ trong văn bản.

Độ thưa tăng lên 99.9351%. Tỷ lệ ô khác 0 còn: 100% − 99.9351% = 0.0649%, tức mỗi văn bản giờ chỉ còn trung bình khoảng 125 từ vựng riêng biệt (giảm từ 170 ở Pipeline A do các từ phổ biến đã bị lọc bỏ).


## 9.3 Pipeline C Extended

In [15]:
tok = Tokenizer.from_pretrained('bert-base-uncased')
encs = tok.encode_batch(texts)
vocab_3 = tok.get_vocab_size()

lengths_c = [len(e.ids) for e in encs]
avg_tok_c = float(np.mean(lengths_c))

/home/trong-thanh/miniconda3/envs/ml/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
indptr = [0]
indices = []
data = []
for e in encs:
    u_ids, counts = np.unique(e.ids, return_counts=True)
    indices.extend(u_ids)
    data.extend(counts)
    indptr.append(len(indices))


In [17]:
X_3 = sp.csr_matrix((data, indices, indptr), shape=(len(texts), vocab_3), dtype=np.int32)
sparsity_c = 1.0 - (X_3.nnz / (X_3.shape[0] * X_3.shape[1]))


In [18]:
print(f'Vocabulary size: {vocab_3}')
print(f'Average token/docs: {avg_tok_c}')
print(f'Matrix shape: {X_3.shape}')
print(f'Matrix sparsity (S): {sparsity_c:.6f} ({sparsity_c * 100:.4f}%)')


Vocabulary size: 30522
Average token/docs: 467.0992
Matrix shape: (30000, 30522)
Matrix sparsity (S): 0.993621 (99.3621%)


Pipeline C sử dụng subword tokenization (WordPiece) với từ điển cố định của mô hình BERT.

Từ điển cố định nhỏ hơn rất nhiều so với A và B, chỉ có 30.522 từ vựng. 

Số token trung bình mỗi văn bản tăng lên ~467 token vì các từ dài hoặc từ hiếm bị bẻ nhỏ thành nhiều subword.

Độ thưa giảm xuống còn 99.3621% (ít thưa nhất trong cả 3 pipeline). Tỷ lệ ô khác 0 là 0.6379%, cao gấp gần 10 lần Pipeline B vì từ điển nhỏ và mỗi văn bản chứa nhiều subword lặp lại.


## 9.4 So sánh 3 pipeline

In [19]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "myocardial infarction cardiology",
    "covid-19 pneumonia coronavirus"
]

In [20]:
tokens_a = [w.lower() for q in queries for w in re.findall(r'\b\w+\b', q)]
oov_a = sum(1 for w in tokens_a if w not in vector_1.vocabulary_) / len(tokens_a) * 100

stop_b = vector_2.get_stop_words()
tokens_b = [w for w in tokens_a if w not in stop_b]
oov_b = sum(1 for w in tokens_b if w not in vector_2.vocabulary_) / len(tokens_b) * 100

unk_id = tok.token_to_id("[UNK]")
encs_q = tok.encode_batch(queries)
total_sub = sum(len(e.ids) for e in encs_q)
oov_c = sum(e.ids.count(unk_id) for e in encs_q) / total_sub * 100


In [21]:
tfidf_1 = TfidfTransformer().fit_transform(X_1)
tfidf_2 = TfidfTransformer().fit_transform(X_2)
tfidf_3 = TfidfTransformer().fit_transform(X_3)
def get_top5_sim(queries, vec, tfidf_mat, is_subword=False):
    res = []
    for q in queries:
        if not is_subword:
            q_v = TfidfTransformer().fit(tfidf_mat).transform(vec.transform([q]))
        else:
            e = tok.encode(q)
            u, cnt = np.unique(e.ids, return_counts=True)
            csr = sp.csr_matrix((cnt, ([0]*len(u), u)), shape=(1, vocab_3), dtype=np.float32)
            q_v = TfidfTransformer().fit(tfidf_mat).transform(csr)
        s = cosine_similarity(q_v, tfidf_mat).flatten()
        res.append(np.mean(np.sort(s)[::-1][:5]))
    return float(np.mean(res))


In [22]:
sim_a = get_top5_sim(queries, vector_1, tfidf_1)
sim_b = get_top5_sim(queries, vector_2, tfidf_2)
sim_c = get_top5_sim(queries, tok, tfidf_3, is_subword=True)
df_compare = pd.DataFrame({
    "Metric": [
        "Vocabulary size",
        "Average tokens/document",
        "Matrix sparsity",
        "OOV rate",
        "Search performance (Top 5 sim)"
    ],
    "Pipeline A": [f"{vocab_1:,}", f"{avg_tok_a:.2f}", f"{sparsity_a:.6f}", f"{oov_a:.2f}%", f"{sim_a:.4f}"],
    "Pipeline B": [f"{vocab_b:,}", f"{avg_tok_b:.2f}", f"{sparsity_b:.6f}", f"{oov_b:.2f}%", f"{sim_b:.4f}"],
    "Pipeline C": [f"{vocab_3:,}", f"{avg_tok_c:.2f}", f"{sparsity_c:.6f}", f"{oov_c:.2f}%", f"{sim_c:.4f}"]
})
display(df_compare)


,Metric,Pipeline A,Pipeline B,Pipeline C
0,Vocabulary size,"193,837","192,764","30,522"
1,Average tokens/document,369.70,197.47,467.10
2,Matrix sparsity,0.999122,0.999351,0.993621
3,OOV rate,5.26%,5.26%,0.00%
4,Search performance (Top 5 sim),0.2499,0.2651,0.2704


## 9.5 Câu hỏi phân tích

### Câu 1

- Thực nghiệm cho thấy khi chưa lowercasing, từ điển có **240.792** từ vựng. Sau khi lowercasing, từ điển giảm xuống còn **193.837** từ (giảm **46.955** từ, tương đương **19.50%**).
- Việc giảm này giúp gom các biến thể, nhưng làm mất phân biệt thực thể: trong dữ liệu gốc tồn tại đồng thời cả hai từ mang nghĩa khác nhau như `US` (nước Mỹ) và `us` (chúng tôi), `Apple` (công ty) và `apple` (quả táo), sau khi lowercasing chúng bị gộp làm một.


In [26]:
text_sample = texts[9]
upper_tokens = [w for w in re.findall(r'\b[A-Z]{2,}\b', text_sample)[:3]]
print("Từ viết hoa gốc trong văn bản:", upper_tokens)
print("Từ sau khi qua vector_1:", [w.lower() for w in upper_tokens])


Từ viết hoa gốc trong văn bản: ['US', 'US', 'US']
Từ sau khi qua vector_1: ['us', 'us', 'us']


### Câu 2

Việc bỏ không phải lúc nào cũng tốt, bỏ stopword giúp văn bản ngắn gọn, tập trung vào từ khóa chính, nhưng sẽ làm mất nghĩa của các câu ngắn, câu thành ngữ (như "to be or not to be") hoặc các cụm từ quan trọng.


In [29]:
q_stop = "to be or not to be"
print(f"Số từ tìm thấy ở Pipeline A: {vector_1.transform([q_stop]).nnz} từ")
print(f"Số từ tìm thấy ở Pipeline B: {vector_2.transform([q_stop]).nnz} từ")


Số từ tìm thấy ở Pipeline A: 4 từ
Số từ tìm thấy ở Pipeline B: 0 từ


### Câu 3

Mất số thập phân (3.14 thành 3 và 14), tên viết tắt, tên công nghệ (C++, C# thành C), email, đường link và sắc thái cảm xúc (!, ?).


Câu gốc: Mac OS X Lion (10.7) and covid-19

Tokens thu được: ['mac', 'os', 'x', 'lion', '10', '7', 'and',  'covid', '19']

10.7 bị tách thành: ['10', '7']

covid-19 bị tách thành: ['covid', '19']

### Câu 4 

Pipeline B thưa nhất (99.9351%) vì vừa có kích thước từ điển lớn vừa bỏ đi các từ xuất hiện nhiều là stop words. Pipeline C ít thưa nhất (99.3621%) do từ điển cố định nhỏ.

### Câu 5

Pipeline B cho kết quả điểm tương đồng cao nhất với các query từ khóa chuẩn vì không bị nhiễu bởi stopword. Pipeline C tốt nhất khi gặp từ mới, từ hiếm hoặc từ sai chính tả vì không bị OOV.

In [34]:
tu = "myocardial"
print("Subword:", tok.encode(tu).tokens)


Subword: ['[CLS]', 'my', '##oca', '##rdial', '[SEP]']


### Câu 6
Không đồng nghĩa. Kích thước từ điển không quyết định chất lượng tìm kiếm:
- Giữa Pipeline A và B, từ điển của B chỉ giảm rất ít (~0.5%) nhưng search tốt hơn rõ rệt nhờ loại bỏ stopword gây nhiễu.
- Nếu cố tình ép từ điển quá nhỏ, các từ khóa chuyên ngành hiếm (như *myocardial*, *infarction*) sẽ bị OOV hoàn toàn, khiến hệ thống không tìm được gì.
- Pipeline C từ điển nhỏ nhất (30.522 từ) nhưng bẻ từ thành subword, dễ gây trùng khớp giả (ví dụ *myocardial* bị bẻ có chứa mảnh *"my"*, vô tình khớp với câu chứa đại từ *"my"*).

In [ ]:
# 1. Từ chuyên ngành nằm sâu trong từ điển, nếu cắt nhỏ từ điển sẽ bị OOV
print("Index của 'myocardial' trong Pipeline B:", vector_2.vocabulary_.get("myocardial"))

# 2. Nguy cơ từ điển subword nhỏ: Trùng mảnh "my" với câu không liên quan
print("Subwords 'myocardial':", tok.encode("myocardial").tokens[1:-1])
print("Subwords 'this is my cat':", tok.encode("this is my cat").tokens[1:-1])
